In [1]:
# Basic libraries 
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

from src.local_config import *
from src.trading_signal import *

In [2]:
# Load in data from previous notebook
df1_is = pd.read_csv(PROJECT_ROOT / "data/df1_is",
                     index_col=0, parse_dates=True)
df1_oos = pd.read_csv(PROJECT_ROOT / "data/df1_oos",
                      index_col=0, parse_dates=True)

df2_is = pd.read_csv(PROJECT_ROOT / "data/df2_is",
                     index_col=0, parse_dates=True)
df2_oos = pd.read_csv(PROJECT_ROOT / "data/df2_oos",
                      index_col=0, parse_dates=True)

static_results_df_is = pd.read_csv(PROJECT_ROOT / "data/static_hedge_ratio_is")
dynamic_results_df_is = pd.read_csv(PROJECT_ROOT / "data/dynamic_hedge_ratio_is")

static_results_df_oos = pd.read_csv(PROJECT_ROOT / "data/static_hedge_ratio_oos")
dynamic_results_df_oos = pd.read_csv(PROJECT_ROOT / "data/dynamic_hedge_ratio_oos")


In [3]:
# Load dictionaries from previous notebook
# In-Sample
with open(PROJECT_ROOT / "data/dictionaries/dynamic_spreads_is.pkl", "rb") as f:
    dynamic_spreads_is = pickle.load(f)
with open(PROJECT_ROOT / "data/dictionaries/dynamic_details_is.pkl", "rb") as f:
    dynamic_details_is = pickle.load(f)

# Out-Of-Sample
with open(PROJECT_ROOT / "data/dictionaries/dynamic_spreads_oos.pkl", "rb") as f:
    dynamic_spreads_oos = pickle.load(f)
with open(PROJECT_ROOT / "data/dictionaries/dynamic_details_oos.pkl", "rb") as f:
    dynamic_details_oos = pickle.load(f)

# Load Portfolio Weighting
portfolio    = pd.read_csv(PROJECT_ROOT / "data/portfolio")
portfolio_pairs = set(portfolio["pair"])

# Implementation - Trading Signal
- For backtest here and compare the Kalman indicator to the baseline and a simple long strategy of only Asset A and Asset B (backtest.py)
- We implement a basic trading signal using standardised values for our dynamic spreads (z-scores)

The general rule is that:
- If the z-score > entry_z - we short the spread 
- If the z-score < -entry_z - we long the spread
- If the absolute z-score < exit_z - we exit our position 
- Otherwise - we hold the previous position

Mathematically, we can express it in a mapping:
$$
\text{Position}(z) =
\begin{cases}
-1 & \text{if } z > z_{\text{entry}} \\
+1 & \text{if } z < -z_{\text{entry}} \\
0 & \text{if } |z| < z_{\text{exit}} \\
\text{hold previous position} & \text{otherwise}
\end{cases}
$$

$z_\text{entry}$ and $z_\text{exit}$ are swept over a grid of candidate values below (rather than fixed at a single value) to select the headline configuration used throughout the rest of the project.

In [4]:
entry_values = [1.5, 2.0, 2.5]
exit_values = [0.0, 0.5, 1.0]

is_results = {}
is_summary_rows = []

for e in entry_values:
    for x in exit_values:
        key = f"entry_{e}_exit_{x}"

        is_signals_dict = generate_kalman_signals(
            dynamic_details_is,
            entry_z=e,
            exit_z=x
        )

        is_results[key] = is_signals_dict

        for (pair_name, R), df in is_signals_dict.items():
            is_summary_rows.append({
                "strategy": key,
                "pair": pair_name,
                "obs_cov": R,
                "entry_z": e,
                "exit_z": x,
                "num_trades": df["position"].diff().abs().sum() / 2,
                "mean_zscore": df["zscore"].mean(),
                "std_zscore": df["zscore"].std()
            })

is_trading_signals = pd.DataFrame(is_summary_rows)

In [5]:
is_summary_rows

[{'strategy': 'entry_1.5_exit_0.0',
  'pair': '285 HK Equity vs 3690 HK Equity',
  'obs_cov': 0.5,
  'entry_z': 1.5,
  'exit_z': 0.0,
  'num_trades': np.float64(24.5),
  'mean_zscore': np.float64(-0.04228639856937039),
  'std_zscore': np.float64(1.1809749821123459)},
 {'strategy': 'entry_1.5_exit_0.0',
  'pair': '285 HK Equity vs 3690 HK Equity',
  'obs_cov': 1.0,
  'entry_z': 1.5,
  'exit_z': 0.0,
  'num_trades': np.float64(20.5),
  'mean_zscore': np.float64(-0.052838447569483096),
  'std_zscore': np.float64(1.2096732841575342)},
 {'strategy': 'entry_1.5_exit_0.0',
  'pair': '285 HK Equity vs 3690 HK Equity',
  'obs_cov': 5.0,
  'entry_z': 1.5,
  'exit_z': 0.0,
  'num_trades': np.float64(20.5),
  'mean_zscore': np.float64(-0.060142679777233685),
  'std_zscore': np.float64(1.260428239182071)},
 {'strategy': 'entry_1.5_exit_0.0',
  'pair': '1818 HK Equity vs 1921 HK Equity',
  'obs_cov': 0.5,
  'entry_z': 1.5,
  'exit_z': 0.0,
  'num_trades': np.float64(15.5),
  'mean_zscore': np.float6

In [6]:
is_trading_signals

,strategy,pair,obs_cov,entry_z,exit_z,num_trades,mean_zscore,std_zscore
0,entry_1.5_exit_0.0,285 HK Equity vs 3690 HK Equity,0.5,1.5,0.0,24.5,-0.042286,1.180975
1,entry_1.5_exit_0.0,285 HK Equity vs 3690 HK Equity,1.0,1.5,0.0,20.5,-0.052838,1.209673
2,entry_1.5_exit_0.0,285 HK Equity vs 3690 HK Equity,5.0,1.5,0.0,20.5,-0.060143,1.260428
3,entry_1.5_exit_0.0,1818 HK Equity vs 1921 HK Equity,0.5,1.5,0.0,15.5,-0.184790,1.248875
4,entry_1.5_exit_0.0,1818 HK Equity vs 1921 HK Equity,1.0,1.5,0.0,15.5,-0.206375,1.255909
...,...,...,...,...,...,...,...,...
76,entry_2.5_exit_1.0,1818 HK Equity vs 1921 HK Equity,1.0,2.5,1.0,8.5,-0.206375,1.255909
77,entry_2.5_exit_1.0,1818 HK Equity vs 1921 HK Equity,5.0,2.5,1.0,7.5,-0.288420,1.249248
78,entry_2.5_exit_1.0,2386 HK Equity vs 386 HK Equity,0.5,2.5,1.0,24.0,-0.036760,1.271678
79,entry_2.5_exit_1.0,2386 HK Equity vs 386 HK Equity,1.0,2.5,1.0,25.0,-0.042370,1.289596


In [7]:
# Save trading_signal details 
with open(PROJECT_ROOT / "data/dictionaries/is_trading_signals.pkl", "wb") as f:
    pickle.dump(is_trading_signals, f) 

with open(PROJECT_ROOT / "data/dictionaries/is_results.pkl", "wb") as f:
    pickle.dump(is_results, f)

# OOS Signals

In [8]:
WARMUP_DAYS = 60  # matches the default z_window in generate_kalman_signals

dynamic_details_portfolio_is = {
    k: v for k, v in dynamic_details_is.items()
    if k[0] in portfolio_pairs
}

dynamic_details_portfolio_oos_raw = {
    k: v for k, v in dynamic_details_oos.items()
    if k[0] in portfolio_pairs
}

dynamic_details_portfolio_oos = {}
oos_start_dates = {}
for k, oos_df in dynamic_details_portfolio_oos_raw.items():
    oos_start_dates[k] = oos_df.index.min()
    is_df = dynamic_details_portfolio_is.get(k)
    if is_df is not None and len(is_df) > 0:
        warmup = is_df.tail(WARMUP_DAYS)
        dynamic_details_portfolio_oos[k] = pd.concat([warmup, oos_df]).sort_index()
    else:
        dynamic_details_portfolio_oos[k] = oos_df

print("IS  pairs:", sorted(set(k[0] for k in dynamic_details_portfolio_is)))
print("OOS pairs:", sorted(set(k[0] for k in dynamic_details_portfolio_oos)))

IS  pairs: ['1818 HK Equity vs 1921 HK Equity', '2386 HK Equity vs 386 HK Equity', '285 HK Equity vs 3690 HK Equity']
OOS pairs: ['1818 HK Equity vs 1921 HK Equity', '2386 HK Equity vs 386 HK Equity', '285 HK Equity vs 3690 HK Equity']


In [9]:
entry_values = [1.5, 2.0, 2.5]
exit_values  = [0.0, 0.5, 1.0]

oos_results      = {}
oos_summary_rows = []

for e in entry_values:
    for x in exit_values:
        key = f"entry_{e}_exit_{x}"

        raw_signals_dict = generate_kalman_signals(
            dynamic_details_portfolio_oos,
            entry_z=e,
            exit_z=x
        )

        # Slice back to true OOS dates: the prepended IS tail only exists to
        # give the rolling z-score/position enough context to be valid from
        # the first real OOS date -- it must never be counted as OOS itself.
        oos_signals_dict = {
            pk: df.loc[df.index >= oos_start_dates[pk]]
            for pk, df in raw_signals_dict.items()
        }

        oos_results[key] = oos_signals_dict

        for (pair_name, R), df in oos_signals_dict.items():
            oos_summary_rows.append({
                "strategy":  key,
                "pair":      pair_name,
                "obs_cov":   R,
                "entry_z":   e,
                "exit_z":    x,
                "num_trades": df["position"].diff().abs().sum() / 2,
                "mean_zscore": df["zscore"].mean(),
                "std_zscore":  df["zscore"].std()
            })

oos_trading_signals = pd.DataFrame(oos_summary_rows)

In [10]:
oos_trading_signals

,strategy,pair,obs_cov,entry_z,exit_z,num_trades,mean_zscore,std_zscore
0,entry_1.5_exit_0.0,285 HK Equity vs 3690 HK Equity,0.5,1.5,0.0,14.0,0.004642,1.219262
1,entry_1.5_exit_0.0,285 HK Equity vs 3690 HK Equity,1.0,1.5,0.0,12.0,0.003216,1.262054
2,entry_1.5_exit_0.0,285 HK Equity vs 3690 HK Equity,5.0,1.5,0.0,13.0,-0.003261,1.336770
3,entry_1.5_exit_0.0,1818 HK Equity vs 1921 HK Equity,0.5,1.5,0.0,15.0,-0.020829,1.296503
4,entry_1.5_exit_0.0,1818 HK Equity vs 1921 HK Equity,1.0,1.5,0.0,15.0,-0.008793,1.324140
...,...,...,...,...,...,...,...,...
76,entry_2.5_exit_1.0,1818 HK Equity vs 1921 HK Equity,1.0,2.5,1.0,7.5,-0.008793,1.324140
77,entry_2.5_exit_1.0,1818 HK Equity vs 1921 HK Equity,5.0,2.5,1.0,9.5,0.107932,1.382905
78,entry_2.5_exit_1.0,2386 HK Equity vs 386 HK Equity,0.5,2.5,1.0,12.5,0.147509,1.283578
79,entry_2.5_exit_1.0,2386 HK Equity vs 386 HK Equity,1.0,2.5,1.0,12.5,0.149945,1.305714


In [11]:
oos_results

{'entry_1.5_exit_0.0': {('285 HK Equity vs 3690 HK Equity',
   0.5):                    y         x   alpha_t    beta_t  spread_t  obs_cov  \
  Date                                                                    
  2022-10-03  2.931727  5.085743  0.107135  0.544861  0.053568      0.5   
  2022-10-04  2.931727  5.085743  0.108126  0.549899  0.026958      0.5   
  2022-10-05  3.022861  5.164214  0.107595  0.554951  0.049380      0.5   
  2022-10-06  2.991724  5.176715  0.107498  0.555534  0.008386      0.5   
  2022-10-07  2.952825  5.152713  0.107490  0.554845 -0.013620      0.5   
  ...              ...       ...       ...       ...       ...      ...   
  2025-12-25  3.513335  4.636669  0.761089  0.599842 -0.029022      0.5   
  2025-12-26  3.513335  4.636669  0.761179  0.599415 -0.027133      0.5   
  2025-12-29  3.515121  4.646312  0.761350  0.598941 -0.029094      0.5   
  2025-12-30  3.530470  4.647271  0.761424  0.598723 -0.013384      0.5   
  2025-12-31  3.515716  4.637637 

In [12]:
# Save trading_signal details 
with open(PROJECT_ROOT / "data/dictionaries/oos_trading_signals.pkl", "wb") as f:
    pickle.dump(oos_trading_signals, f) 

with open(PROJECT_ROOT / "data/dictionaries/oos_results.pkl", "wb") as f:
    pickle.dump(oos_results, f)